<a href="https://colab.research.google.com/github/kavishkat2002/flyrank-ml-kavishka/blob/main/w02_ml_task_framing.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# ML-03 — Frame Your Lane as an ML Task

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/flyrank-bih/flyrank-ml-internship-starter/blob/main/work/notebooks/w02_ml_task_framing.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

**Lane chosen:** Lane 2 — Refresh / Content Opportunity Scoring.


## 1. My lane as an ML task (type)

*Classification, clustering, ranking, or scoring — which one, and why?*

My lane is **Refresh / Content Opportunity Scoring**. The decision it improves is: *which pages should
a content editor review first this week, out of thousands?*

That question is a **ranking / scoring** task, not a plain classification task. An editor does not
want a single yes/no verdict per page ("declining" vs "not declining") — they want a **priority
order**: a queue of the pages most worth their limited review time, from most to least urgent.

Underneath the ranking sits one supporting classification signal — a binary label for whether a
page is currently declining (`is_declining_label`, built from the observed `trend_direction`
column). That label is one input into the priority score, not the deliverable itself. The
deliverable is the **ordered queue**, so I'm framing this as scoring/ranking with a classifier as
one ingredient — matching what `docs/ml-intern-dataset-and-lane-guide.md` calls out for Lane 2:
"a ranked queue with reason codes."


In [4]:
# Clone the GitHub repository to access the data files.
!git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git

Cloning into 'flyrank-ml-internship-starter'...
remote: Enumerating objects: 283, done.
remote: Counting objects: 100% (145/145), done.
remote: Compressing objects: 100% (62/62), done.
remote: Total 283 (delta 112), reused 83 (delta 83), pack-reused 138 (from 1)
Receiving objects: 100% (283/283), 1.85 MiB | 4.99 MiB/s, done.
Resolving deltas: 100% (153/153), done.


In [9]:
# Confirms the framing choice against the actual label the pipeline defines.
import pandas as pd

# Please ensure you run !git clone https://github.com/flyrank-bih/flyrank-ml-internship-starter.git
# in a preceding cell. The file path has been updated assuming the repository is cloned
# into the Colab environment's root directory.
df = pd.read_csv("flyrank-ml-internship-starter/data/raw/content_refresh_anonymized.csv")

print(f"Rows: {len(df):,} | Columns: {df.shape[1]}")
print()
print("trend_direction (source of the label) value counts:")
print(df["trend_direction"].value_counts())
print()
declining_share = (df["trend_direction"] == "down").mean() * 100
print(f"Share currently 'down' (would be label=1): {declining_share:.1f}%")

Rows: 30,000 | Columns: 44

trend_direction (source of the label) value counts:
trend_direction
down      16262
stable     5962
up         4388
new        2236
flat       1152
Name: count, dtype: int64

Share currently 'down' (would be label=1): 54.2%


In [5]:
# Verify the contents of the data/raw directory within the cloned repository
!ls -F flyrank-ml-internship-starter/data/raw/

content_refresh_anonymized.csv


## 2. Target or proxy

*What would you predict? Where does that label come from — observed outcome or a defined rule?*

**Target:** `is_declining_label` — 1 if a page's `trend_direction` is `"down"`, else 0. Per the
data dictionary, `trend_direction` is itself computed from `trend_pct`, which compares
`impressions_last_30d` to `impressions_prev_30d` (an actual measured swing in search impressions,
not something I invented). So this is an **observed outcome**, not a hand-defined rule — it
reflects real traffic movement that already happened, which is exactly what the framing skill
calls the safer kind of target ("prefer outcomes measured in a later time window").

**Important leakage note:** `trend_direction` and `trend_pct` are the columns the label is *built
from*, so they can never also be model features — that would let the model see its own answer.
The final priority score is really a **combination**: `P(declining)` from the classifier, blended
with the page's `impression_tier` / `avg_position` context to produce the actual ranked queue —
that blended number is the "scoring" part of the lane, and the classifier's probability is the
proxy that feeds it.


In [10]:
# Show where the label comes from and confirm it isn't a feature-derived shortcut.
df["is_declining_label"] = (df["trend_direction"] == "down").astype(int)

print(df[["trend_pct", "trend_direction", "is_declining_label"]].sample(8, random_state=7))
print()
print(f"Positive rate (label=1): {df['is_declining_label'].mean()*100:.1f}%")

       trend_pct trend_direction  is_declining_label
1252        -6.0          stable                   0
10444      -91.9            down                   1
8994       -18.2          stable                   0
7463         NaN            flat                   0
1910       -69.6            down                   1
20606      133.3              up                   0
9866         7.5          stable                   0
3737        86.7              up                   0

Positive rate (label=1): 54.2%


## 3. Success metric

*One metric you can defend. What number means 'good'?*

**Primary metric: Precision@50** — of the top 50 pages the model puts at the front of the queue,
what fraction are actually declining? An editor only has time to review a handful of pages a
week, so what matters is whether the *top of the list* is worth their time — not overall accuracy
across all 30,000 pages, most of which nobody will ever look at.

Precision@50 is the metric because:
- It matches the real action (an editor works top-down through a short queue, not the whole file).
- It's directly comparable to a naive baseline (e.g., "flag every page with `impression_tier`
  ≥ `moderate`"), so I can show the model is worth building at all.
- It ignores the huge pool of true negatives that classification accuracy would let the model
  get "credit" for by just predicting the majority case.

Secondary metric I'll also report: **recall at the same cutoff**, so I can see how much real
decline the top-50 queue is missing — useful context, not the headline number.


In [11]:
# Sketch what "good" looks like using the two extremes: a random queue and a perfect queue.
import numpy as np

K = 50
rng = np.random.default_rng(7)

base_rate = df["is_declining_label"].mean()

# average over many random draws, since a single draw of 50 is noisy
trials = 2000
precisions = [rng.choice(df["is_declining_label"].values, size=K, replace=False).mean() for _ in range(trials)]
precision_at_k_random = np.mean(precisions)

# best-case ceiling: if the top 50 were perfectly chosen from the positive pool
precision_at_k_perfect = 1.0 if df["is_declining_label"].sum() >= K else df["is_declining_label"].sum() / K

print(f"Base decline rate across all pages:        {base_rate*100:.1f}%")
print(f"Precision@{K} from a RANDOM queue (avg of {trials} draws): {precision_at_k_random*100:.1f}%  (what 'no model' looks like)")
print(f"Precision@{K} from a PERFECT queue:                {precision_at_k_perfect*100:.1f}%  (the ceiling)")
print()
print("A random queue's expected precision just tracks the base rate. 'Good' for this project means")
print("beating that baseline by a wide, defensible margin - not chasing the ceiling, which would mean")
print("the label was trivial to predict in the first place.")

Base decline rate across all pages:        54.2%
Precision@50 from a RANDOM queue (avg of 2000 draws): 54.6%  (what 'no model' looks like)
Precision@50 from a PERFECT queue:                100.0%  (the ceiling)

A random queue's expected precision just tracks the base rate. 'Good' for this project means
beating that baseline by a wide, defensible margin - not chasing the ceiling, which would mean
the label was trivial to predict in the first place.


## 4. The unit of analysis, as a real dataframe

*Load your lane's slice and show it: one row = one what?*

**One row = one content item (page)**, identified by `content_id`, belonging to one `client_id`,
with its trailing-90-day search and engagement metrics. This is the natural unit for the lane
because the decision ("review this page or not") is made per page — not per client, not per
query, not per day.


In [12]:
# My lane's slice: the columns an editor-facing refresh queue actually needs —
# identity, the safe leakage-free features, and the label (kept separate, never fed back in as a feature).
lane_cols = [
    "content_id", "client_id", "content_type", "main_intent",
    "impressions_90d", "clicks_90d", "ctr", "avg_position", "position_tier", "impression_tier",
    "word_count", "content_age_days", "days_since_last_update", "freshness_tier",
    "engagement_rate", "ai_traffic_pct",
    "is_declining_label",   # target — observed, not a feature
]

lane_df = df[lane_cols].copy()

print(f"Unit of analysis: one row = one content item (page). Shape: {lane_df.shape}")
lane_df.head(10)

Unit of analysis: one row = one content item (page). Shape: (30000, 17)


,content_id,client_id,content_type,main_intent,impressions_90d,clicks_90d,ctr,avg_position,position_tier,impression_tier,word_count,content_age_days,days_since_last_update,freshness_tier,engagement_rate,ai_traffic_pct,is_declining_label
0,content_304f48230142,client_f369cb89fc,keyword article,transactional,3803,29,0.76,10.6,striking,good,3221.0,187,20,0-30,5.88,0.0,1
1,content_a1fb4e703a9e,client_4e07408562,keyword article,informational,15320,7,0.05,20.3,page_3_5,good,2481.0,445,25,0-30,0.00,0.0,1
2,content_9aa793d4d895,client_7f2253d7e2,keyword article,informational,12581,11,0.09,36.5,page_3_5,good,3515.0,141,20,0-30,0.00,0.0,1
3,content_331d6c4de07b,client_19581e27de,keyword article,commercial,11751,58,0.49,6.2,page_1,good,NaN,463,22,0-30,1.28,0.0,0
4,content_d99b7a2d90ca,client_3fdba35f04,keyword article,informational,19140,24,0.13,44.0,page_3_5,good,2803.0,263,14,0-30,0.00,0.0,1
5,content_d4084a4bc775,client_f369cb89fc,keyword article,transactional,3970,1,0.03,8.5,page_1,good,3080.0,147,20,0-30,0.00,0.0,1
6,content_9a34b442b552,client_8722616204,keyword article,informational,20,0,0.00,7.0,page_1,low,3059.0,90,20,0-30,0.00,0.0,1
7,content_a63219c6e95a,client_19581e27de,keyword article,commercial,1724,1,0.06,21.2,page_3_5,moderate,NaN,445,22,0-30,3.57,0.0,0
8,content_5e6c160719bc,client_6208ef0f77,keyword article,informational,32574,29,0.09,46.0,page_3_5,excellent,3807.0,90,20,0-30,5.88,0.0,1
9,content_c27558df2b0c,client_19581e27de,keyword article,informational,1240,2,0.16,4.9,page_1,moderate,NaN,257,104,91-180,0.00,0.0,1


In [14]:
# Sanity check: content_id really is unique per row (confirms the grain).
print("Unique content_id count:", lane_df["content_id"].nunique(), "vs total rows:", len(lane_df))
print("Unique client_id count:", lane_df["client_id"].nunique(), "(clients represented in this slice)")

Unique content_id count: 30000 vs total rows: 30000
Unique client_id count: 32 (clients represented in this slice)


## 5. Why ML beats a fixed rule here

*What makes the pattern too messy for an if-statement?*

A fixed rule *can* flag obvious cases — e.g., "if `trend_pct < -20%`, flag it" — but that's
circular: it's just restating the label with different words, and it can't be applied before the
trend is already known and over. What editors actually need is a **priority score built from
signals available *before* or *alongside* the decline**, and that's where a single if-statement
breaks down:

- **Many weakly-informative signals, not one strong one.** `impression_tier`, `avg_position`,
  `content_age_days`, `days_since_last_update`, `engagement_rate`, `content_type`, and
  `main_intent` all carry a little bit of information about risk, but no single threshold on any
  one of them cleanly separates decliners from non-decliners.
- **Interactions matter.** A stale page (`freshness_tier = "never"` region) with high impressions
  is a very different risk than a stale page with almost no traffic — a fixed rule would need a
  combinatorial explosion of nested if/else branches to capture that, and it would still miss
  interactions nobody thought to hand-code.
- **The relationship isn't linear or stable.** Risk doesn't rise in neat steps as position
  worsens or age increases — a model can fit the actual curve instead of a human guessing at
  breakpoints.
- **It has to keep working as the data shifts.** New content types, new clients, and seasonal
  swings change the mix constantly; an if-statement has to be manually rewritten each time, while
  a model can be retrained on fresh data.

I'll confirm this isn't just an assumption — this is exactly the leakage-safe A/B comparison the
lane calls for: rule baseline vs. model, both scored with Precision@50, in a later notebook.


In [15]:
# Quick evidence check now: does a single reasonable threshold rule already separate the classes cleanly?
# If it did, ML would be unnecessary. Check with 'engagement_rate' as one candidate rule signal.
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt

fig, ax = plt.subplots(figsize=(6, 4))
for label, group in lane_df.groupby("is_declining_label"):
    ax.hist(group["engagement_rate"].dropna(), bins=30, alpha=0.5,
            label=f"is_declining_label={label}", density=True)
ax.set_xlabel("engagement_rate")
ax.set_ylabel("density")
ax.set_title("engagement_rate distribution by label — heavy overlap, no clean cutoff")
ax.legend()
plt.tight_layout()
plt.savefig("engagement_rate_overlap.png", dpi=100)
plt.show()
print("Saved chart: engagement_rate_overlap.png")
print()
print("The two distributions overlap heavily — no single threshold on this signal (or most others")
print("checked individually) cleanly separates decliners from non-decliners. That overlap is the")
print("concrete evidence that this needs a model combining several signals, not one if-statement.")

Saved chart: engagement_rate_overlap.png

The two distributions overlap heavily — no single threshold on this signal (or most others
checked individually) cleanly separates decliners from non-decliners. That overlap is the
concrete evidence that this needs a model combining several signals, not one if-statement.


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere (only pseudonymous `content_id` / `client_id`)
- [x] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.
